# 🚀 DSPy Starter: Building & Optimizing AI Systems

Unlike traditional prompt engineering, **DSPy** allows you to define declarative modules (like building neural networks) and uses optimizers to automatically compile and rewrite your prompts to maximize accuracy. 

In this starter, we will:
1. Configure an OpenAI Language Model.
2. Define a custom Question-Answering task using `dspy.Signature`.
3. Provide a small training dataset.
4. Use a DSPy **Optimizer** to automatically write the best prompt for us!

In [2]:
import os
import dspy
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    print("⚠️ WARNING: OPENAI_API_KEY not found in .env file.")

# Configure DSPy to use GPT-4o-Mini as the default language model
lm = dspy.LM('openai/gpt-4o-mini', max_tokens=1000)
dspy.configure(lm=lm)

print("✅ DSPy configured successfully!")

✅ DSPy configured successfully!


## 1. Define the Signature & Module
A **Signature** defines the inputs and outputs of your task (the *what*).
A **Module** defines the execution pipeline (the *how*). We will use DSPy's built-in `ChainOfThought` module to give the AI reasoning capabilities.

In [3]:
# Define the I/O behavior for our AI system
class BasicQA(dspy.Signature):
    """Answer questions with short, precise, and accurate fact-based answers."""
    
    question = dspy.InputField(desc="The user's query.")
    answer = dspy.OutputField(desc="A concise, factual answer.")

# Build the execution module
class CoTQA(dspy.Module):
    def __init__(self):
        super().__init__()
        # Wrap our signature in a Chain of Thought reasoning module
        self.prog = dspy.ChainOfThought(BasicQA)
        
    def forward(self, question):
        return self.prog(question=question)

# Let's test the UN-OPTIMIZED system
unoptimized_agent = CoTQA()
response = unoptimized_agent(question="What is the capital of France?")

print("--- Unoptimized Zero-Shot Response ---")
print(f"Answer: {response.answer}")

--- Unoptimized Zero-Shot Response ---
Answer: Paris


## 2. Optimizing the System
Now for the magic. We will create a small dataset of 5 examples. Then, we will use DSPy's `BootstrapFewShot` optimizer. It will automatically run the module, evaluate the reasoning, and **rewrite the prompt** to include the best few-shot examples that maximize accuracy.

In [8]:
from dspy.teleprompt import BootstrapFewShot

# 1. Create a tiny training dataset (Inputs must be marked as .with_inputs())
trainset = [
    dspy.Example(question="What is the freezing point of water in Celsius?", answer="0 degrees Celsius").with_inputs("question"),
    dspy.Example(question="Who wrote 'To Kill a Mockingbird'?", answer="Harper Lee").with_inputs("question"),
    dspy.Example(question="What is the chemical symbol for Gold?", answer="Au").with_inputs("question"),
    dspy.Example(question="How many planets are in the Solar System?", answer="8").with_inputs("question"),
    dspy.Example(question="What year did the Apollo 11 moon landing occur?", answer="1969").with_inputs("question"),
]

# 2. Define a simple exact-match evaluation metric
def exact_match_metric(example, pred, trace=None):
    return example.answer.lower() in pred.answer.lower()

# 3. Setup the Optimizer (Teleprompter)
optimizer = BootstrapFewShot(
    metric=exact_match_metric,
    max_bootstrapped_demos=3, # The number of optimized examples to inject into the final prompt
    max_labeled_demos=3
)

print("Compiling and optimizing the AI system. Please wait...")

# 4. Compile! DSPy is now writing and optimizing the prompt for you.
optimized_agent = optimizer.compile(student=CoTQA(), trainset=trainset)

print("✅ System successfully optimized!")

Compiling and optimizing the AI system. Please wait...


 80%|████████  | 4/5 [00:00<00:00, 11.49it/s]

Bootstrapped 3 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
✅ System successfully optimized!


In [9]:
# Test the optimized agent on a brand new question
test_question = "Who painted the Mona Lisa?"
optimized_response = optimized_agent(question=test_question)

print(f"Question: {test_question}")
# 👇 Add this line right here to reveal the hidden Chain of Thought!
print(f"Reasoning: {optimized_response.reasoning}") 
print(f"Optimized Answer: {optimized_response.answer}\n")

# THE REVEAL: Let's look at the exact prompt DSPy wrote for us!
print("--- The Prompt DSPy Wrote Under The Hood ---")
lm.inspect_history(n=1)

Question: Who painted the Mona Lisa?
Reasoning: The Mona Lisa is a renowned painting created during the Renaissance period. It was painted by the Italian artist Leonardo da Vinci, and it is famous for its exquisite detail and the subject's enigmatic expression.
Optimized Answer: Leonardo da Vinci

--- The Prompt DSPy Wrote Under The Hood ---




[2026-03-10T18:06:02.084233]

System message:

Your input fields are:
1. `question` (str): The user's query.
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): A concise, factual answer.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer questions with short, precise, and accurate fact-based answers.


User message:

[[ ## question ## ]]
Who wrote 'To Kill a Mockingbird'?


Assistant message:

[[ ## re